# Publication year trends

Summarize yearly publication volume and explicit UK Biobank mentions from the full-endpoint publication parquet.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_06_year_trends")
yearly = (
    df.dropna(subset=["analysis_year"])
    .groupby("analysis_year")
    .agg(
        n_publications=("id", "count"),
        n_explicit_ukb_mentions=("explicit_ukb_mention", "sum"),
    )
    .reset_index()
)
yearly["n_without_explicit_ukb_mention"] = yearly["n_publications"] - yearly["n_explicit_ukb_mentions"]
yearly["explicit_ukb_mention_percent"] = yearly["n_explicit_ukb_mentions"] / yearly["n_publications"] * 100
yearly.to_csv(table_dir / "publication_year_trends.csv", index=False)


In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
axes[0].plot(yearly["analysis_year"], yearly["n_publications"], marker="o")
axes[0].set(ylabel="Publications", title="UK Biobank publication corpus by year")
axes[1].plot(yearly["analysis_year"], yearly["explicit_ukb_mention_percent"], marker="o")
axes[1].set(xlabel="Publication year", ylabel="Explicit UKB mention (%)")
for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
save_figure(figure, figure_dir / "publication_year_trends.png")
yearly.tail(10)
